In [2]:
!pip install pandas numpy scikit-learn tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 32.0 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 31.6 MB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 46.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 54.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 53.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 46.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 58.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 [tensorflow]9 [tensorflow]]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Load dataset
df = pd.read_csv("environment_data_feb2025.csv")

# Convert timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Sort (just in case)
df = df.sort_values("timestamp")

# Select features
features = ["temperature_C", "humidity_%", "AQI"]
data = df[features].values

# Scale data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

print("Shape:", scaled_data.shape)

Shape: (672, 3)


In [4]:
SEQ_LENGTH = 72  # 72 hours

X = []
y = []

for i in range(len(scaled_data) - SEQ_LENGTH):
    X.append(scaled_data[i:i+SEQ_LENGTH])
    y.append(scaled_data[i+SEQ_LENGTH])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)  # (samples, 72, features)
print("y shape:", y.shape)

X shape: (600, 72, 3)
y shape: (600, 3)


In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQ_LENGTH, X.shape[2])),
    LSTM(32),
    Dense(16, activation="relu"),
    Dense(3)  # 3 outputs (temp, humidity, AQI)
])

model.compile(
    optimizer="adam",
    loss="mse"
)

model.summary()

# Train
history = model.fit(
    X, y,
    epochs=10,
    batch_size=16,
    validation_split=0.1
)

I0000 00:00:1776315637.672135    4377 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776315638.027793    4377 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776315640.024176    4377 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1776315640.631926    4377 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 72, 64)         │        17,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,403 (118.76 KB)

 Trainable params: 30,403 (118.76 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - loss: 0.1936 - val_loss: 0.1310
Epoch 2/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.1053 - val_loss: 0.0951
Epoch 3/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.0824 - val_loss: 0.0818
Epoch 4/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0767 - val_loss: 0.0778
Epoch 5/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 0.0749 - val_loss: 0.0744
Epoch 6/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0661 - val_loss: 0.0620
Epoch 7/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0601 - val_loss: 0.0595
Epoch 8/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0578 - val_loss: 0.0599
Epoch 9/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.0599 - val_loss: 0.0578
Epoch 10/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0578 - val_loss: 0.0596


In [7]:
from datetime import datetime, timedelta
import numpy as np

def predict_for_datetime(target_datetime, model, scaler, original_data, seq_length=72):
    """
    target_datetime: string → "2025-03-01 10:00:00"
    """

    target_datetime = datetime.strptime(target_datetime, "%Y-%m-%d %H:%M:%S")

    # Last timestamp in dataset
    last_known_time = datetime(2025, 2, 28, 23, 0, 0)

    if target_datetime <= last_known_time:
        print("❌ Date must be in the future bro")
        return

    # Steps to predict
    hours_ahead = int((target_datetime - last_known_time).total_seconds() / 3600)

    print(f"Predicting {hours_ahead} hours into future...")

    # Start from last 72 hours
    current_seq = original_data[-seq_length:].copy()

    for _ in range(hours_ahead):
        input_seq = current_seq.reshape(1, seq_length, 3)

        pred = model.predict(input_seq, verbose=0)[0]

        # slide window
        current_seq = np.vstack((current_seq[1:], pred))

    # Convert back to real values
    final_pred = scaler.inverse_transform([pred])[0]

    print(f"\n📅 Prediction for {target_datetime}:")
    print(f"🌡️ Temp: {final_pred[0]:.2f} °C")
    print(f"💧 Humidity: {final_pred[1]:.2f} %")
    print(f"🌫️ AQI: {final_pred[2]:.2f}")

    return final_pred

In [10]:
predict_for_datetime(
    "2025-03-01 10:00:00",
    model,
    scaler,
    scaled_data
)

Predicting 11 hours into future...



📅 Prediction for 2025-03-01 10:00:00:
🌡️ Temp: 31.67 °C
💧 Humidity: 64.93 %
🌫️ AQI: 147.33


array([ 31.67160706,  64.92977652, 147.33297229])

In [11]:
import joblib

# Save LSTM model
model.save("lstm_env_model.h5")

# Save scaler (VERY IMPORTANT)
joblib.dump(scaler, "scaler.save")

print("✅ Model and scaler saved successfully!")

✅ Model and scaler saved successfully!
